# DPO with a fixed reference policy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/direct_preference_optimisation.ipynb)

This notebook runs on CPU. It generates all data and performs the experiment; no model download or API credentials are needed. GPU timing is optional for attention.

Dependencies: Python 3.10+, torch, numpy, scipy, matplotlib. In Colab these are normally installed. If needed run `%pip install torch numpy scipy matplotlib` before the imports.

Tested on Python 3.12 with PyTorch 2.14.0+rocm10.1.0, NumPy 2.5.3, SciPy 1.18.1 and Matplotlib 3.11.2. The ROCm suffix identifies the development machine build; retain the PyTorch build supplied by Colab for its CPU or NVIDIA runtime. Different library versions can change timings and floating-point details.

In [ ]:
import math, time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
torch.set_num_threads(4)

## Implementation

The functions below are the complete implementation. Fixed seeds make the inputs repeatable; timings depend on the machine. Read the accompanying tutorial for derivations and limits.

In [ ]:
def sequence_logp(logits,tokens,mask):
    per_token=logits.log_softmax(-1).gather(-1,tokens.unsqueeze(-1)).squeeze(-1)
    return (per_token*mask).sum(-1)

def dpo_loss(policy_chosen,policy_rejected,ref_chosen,ref_rejected,beta):
    margin=(policy_chosen-policy_rejected)-(ref_chosen-ref_rejected)
    return -F.logsigmoid(beta*margin).mean()

def preference_data(seed,n=4096):
    g=torch.Generator().manual_seed(seed);ctx=torch.randint(12,(n,),generator=g);a=torch.randint(4,(n,),generator=g);b=(a+torch.randint(1,4,(n,),generator=g))%4
    rewards=torch.tensor([[1.2*np.sin(c*.7+j*1.3)+.5*np.cos(c+j) for j in range(4)] for c in range(12)])
    first=torch.rand(n,generator=g)<torch.sigmoid(rewards[ctx,a]-rewards[ctx,b]);chosen=torch.where(first,a,b);rejected=torch.where(first,b,a)
    return ctx,chosen,rejected,rewards

def dpo_experiment():
    start=time.perf_counter();ctx,ch,rj,rewards=preference_data(120);tc,ta,tb,_=preference_data(999,4096)
    ref=torch.log_softmax(torch.tensor([[.6*np.cos(c+j*.8) for j in range(4)] for c in range(12)]),-1)
    initial=float(dpo_loss(ref[ctx,ch],ref[ctx,rj],ref[ctx,ch],ref[ctx,rj],.5));assert abs(initial-math.log(2))<1e-6
    test=torch.tensor([0.,0.],requires_grad=True);loss=dpo_loss(test[0],test[1],torch.tensor(0.),torch.tensor(0.),.5);loss.backward();assert test.grad[0]<0 and test.grad[1]>0
    mask_logits=torch.randn(2,3,4);tokens=torch.tensor([[1,2,0],[3,1,2]]);mask=torch.tensor([[1,1,0],[1,0,0]])
    lp=sequence_logp(mask_logits,tokens,mask);changed=mask_logits.clone();changed[mask==0]=100.;assert torch.allclose(lp,sequence_logp(changed,tokens,mask))
    records=[];traces=[]
    for seed in [0,1,2]:
      for beta in [.1,.5,1.]:
        torch.manual_seed(seed);logits=nn.Parameter(ref.clone());fixed=ref.clone();opt=torch.optim.Adam([logits],lr=.03);history=[]
        for step in range(600):
          idx=torch.randint(len(ctx),(256,));lp=logits.log_softmax(-1);c=ctx[idx];a=ch[idx];b=rj[idx]
          loss=dpo_loss(lp[c,a],lp[c,b],fixed[c,a],fixed[c,b],beta);opt.zero_grad();loss.backward();opt.step()
          if step%30==0:history.append([step,float(loss.detach())])
        with torch.no_grad():
          lp=logits.log_softmax(-1);ratio=lp-fixed;acc=float((ratio[tc,ta]>ratio[tc,tb]).float().mean());kl=float((lp.exp()*(lp-fixed)).sum(-1).mean());reward=float((lp.exp()*rewards).sum(-1).mean());base=float((ref.exp()*rewards).sum(-1).mean())
          assert torch.equal(fixed,ref)
          records.append(dict(seed=seed,beta=beta,heldout_accuracy=acc,kl=kl,expected_reward=reward,reference_reward=base,probabilities=lp.exp().tolist()))
          traces.append(dict(seed=seed,beta=beta,history=history))
    return dict(initial_loss=initial,gradient=test.grad.tolist(),mask_check=True,reference_frozen=True,records=records,traces=traces,seconds=time.perf_counter()-start)

## Experiment and checks

Assertions check correctness before the comparison. The sparse-autoencoder experiment uses held-out activations, and DPO evaluates fresh preference labels on known contexts. Training settings are deliberately small.

In [ ]:
result = dpo_experiment()
print('Elapsed seconds:', result['seconds'])
print(result.get('errors', result['records'][0]))

## Measured figures

Each point comes from the experiment above. For attention on CPU the memory panel is labelled analytical score storage, because CPU allocator peaks are not measured. GPU results use allocator statistics.

In [ ]:
def dpo_plots(result):
    fig,axes=plt.subplots(1,3,figsize=(12,4))
    for beta in [.1,.5,1.]:
      rs=[r for r in result['records'] if r['beta']==beta]
      axes[0].scatter([beta]*3,[r['heldout_accuracy'] for r in rs]);axes[1].scatter([beta]*3,[r['kl'] for r in rs]);axes[2].scatter([beta]*3,[r['expected_reward'] for r in rs])
    for ax,y in zip(axes,['Held-out preference accuracy','Mean KL(policy || reference)','Expected synthetic reward']):ax.set(xlabel='Beta',ylabel=y)
    axes[0].set_ylim(.5,.8);axes[2].axhline(result['records'][0]['reference_reward'],ls='--',color='grey',label='Reference');axes[2].legend();fig.tight_layout();return fig

dpo_plots(result)
plt.show()

## Exercises

1. Change the reference distribution and inspect the resulting policy.
2. Compare the learned policy with softmax(log(reference) + reward / beta).
3. Introduce inconsistent preference labels and measure the attainable accuracy.
4. Add prompt and padding masks to a sequence model, retaining only response-token log-probabilities.

## Reference and scope

Primary reference: https://arxiv.org/abs/2305.18290

This is a controlled teaching experiment. The attention implementation uses ordinary PyTorch operations; the autoencoder uses planted synthetic features; DPO uses categorical responses rather than a language model.